# Resources and mechanisms

Read Chapters 23–31 and Primer-PY first. This notebook executes author-assigned examples, not a hardware benchmark. Independent fractions and integer products are established in the chapters and `expected.json`; this run does not rewrite them. Text: CC BY-SA 4.0. Code: Apache-2.0.

In [1]:
from pathlib import Path
import json
import sys
import importlib.util
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
def load_json(path):
    return json.loads((ROOT / path).read_text())
def module(name, path):
    spec = importlib.util.spec_from_file_location(name, ROOT / path)
    value = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(value)
    return value
sys.path.insert(0, str(ROOT / "code/part-v"))
import math
import mechanisms as m
runner = module("part_v_runner", "code/part-v/run.py")
actual = runner.run()
expected = load_json("data/part-v/expected.json")
assert actual["budget_flops"] == [6 * 100_000_000 * 2_000_000_000] * 3
assert actual["training"]["state_bytes"] == 16_000_000
assert actual["training"]["activation_bytes"] == 2 * 128 * 64 * 4 * 10 * 2
assert actual["pipeline"]["stage_counts"] == [8, 7, 6, 6, 5]
print("Assigned training ledger:", actual["training"])
print("Filter audit:", actual["pipeline"]["ledger"])


Assigned training ledger: {'state_categories': [2000000, 2000000, 4000000, 8000000], 'state_bytes': 16000000, 'activation_bytes': 1310720, 'total_bytes': 17310720, 'with_headroom_bytes': 21638400.0, 'weighted_gradient': 2.5}
Filter audit: [{'id': 'new', 'parsed': 'Beijing: 750', 'code_points': 12, 'digit_fraction': 0.25, 'bad_filter_rejects': True, 'revised_result': 'retained', 'duplicate_of': None, 'redacted': 'Beijing: 750'}, {'id': 'copy', 'parsed': 'Beijing: 750', 'code_points': 12, 'digit_fraction': 0.25, 'bad_filter_rejects': True, 'revised_result': 'exact_duplicate', 'duplicate_of': 'new', 'redacted': 'Beijing: 750'}, {'id': 'old', 'parsed': 'Archived Beijing lodging limit: 600 CNY.', 'code_points': 40, 'digit_fraction': 0.075, 'bad_filter_rejects': False, 'revised_result': 'retained', 'duplicate_of': None, 'redacted': 'Archived Beijing lodging limit: 600 CNY.'}, {'id': 'menu', 'parsed': 'Home | Help | Sign in', 'code_points': 21, 'digit_fraction': 0.0, 'bad_filter_rejects': Tru

In [2]:
assert actual["rotation_dots"] == [1, 0, 0, -1]
assert math.isclose(sum(x*x for x in actual["rms"])/2, 1)
assert math.isclose(actual["swiglu"]["output"][0], (2*math.e-3)/(1+math.e))
assert actual["architecture"]["active_parameters"] == 40_000_000
assert actual["architecture"]["total_parameters"] == 60_000_000
assert actual["rescue"] == [1, 0]
assert m.rescue(2, 2, 5) is None
print("Component outputs:", actual["rms"], actual["swiglu"])
print("Intervention evidence:", actual["intervention"])


Component outputs: [0.848528137423857, 1.131370849898476] {'gate': [1, -1], 'up': [2, 3], 'silu': [0.7310585786300049, -0.2689414213699951], 'product': [1.4621171572600098, -0.8068242641099853], 'output': [0.6552928931500245, -0.8068242641099853]}
Intervention evidence: [{'hidden': [1, 1], 'logits': [2, -2], 'margin': 4}, {'hidden': [-1, -1], 'logits': [-2, 2], 'margin': -4}, {'hidden': [1, -1], 'logits': [2, -2], 'margin': 4}, {'hidden': [-1, 1], 'logits': [-2, 2], 'margin': -4}, {'hidden': [-1, 1], 'logits': [-2, 2], 'margin': -4}, {'hidden': [1, -1], 'logits': [2, -2], 'margin': 4}]


In [3]:
z = [math.log(4), math.log(2), 0, 0]
assert m.distribution(z, top_p=.75) == [2/3, 1/3, 0, 0]
assert m.distribution(z, top_k=2, top_p=.6) == [1, 0, 0, 0]
assert actual["beam"]["2"][0]["tokens"] == ["B", "EOS"]
assert actual["small_cache_bytes"] == [512, 576, 640]
assert actual["large_cache_bytes"] == 384 * 1024**2
assert actual["inference"]["tpot_ms"] == 40
assert m.latency(0, [100])["tpot_ms"] is None
assert m.quantize([8], .5)["errors"] == [-4.5]
print("Sampling matrix:", actual["sampling"])
print("Assigned runtime and serving traces:", actual["inference"], actual["serving"])


Sampling matrix: [{'policy': 'greedy', 'probabilities': [1.0, 0.0, 0.0, 0.0], 'selections': [0, 0, 0, 0, 0, 0]}, {'policy': 'temperature-1', 'probabilities': [0.5, 0.25, 0.125, 0.125], 'selections': [0, 0, 1, 1, 2, 3]}, {'policy': 'temperature-2', 'probabilities': [0.3693980625181293, 0.2612038749637415, 0.18469903125906464, 0.18469903125906464], 'selections': [0, 1, 1, 2, 2, 3]}, {'policy': 'top-k-2', 'probabilities': [0.6666666666666666, 0.3333333333333333, 0.0, 0.0], 'selections': [0, 0, 0, 1, 1, 1]}, {'policy': 'top-p-0.5', 'probabilities': [1.0, 0.0, 0.0, 0.0], 'selections': [0, 0, 0, 0, 0, 0]}, {'policy': 'top-p-0.6', 'probabilities': [0.6666666666666666, 0.3333333333333333, 0.0, 0.0], 'selections': [0, 0, 0, 1, 1, 1]}, {'policy': 'top-p-0.75', 'probabilities': [0.6666666666666666, 0.3333333333333333, 0.0, 0.0], 'selections': [0, 0, 0, 1, 1, 1]}, {'policy': 'top-p-0.8', 'probabilities': [0.5714285714285714, 0.2857142857142857, 0.14285714285714285, 0.0], 'selections': [0, 0, 0, 1,

In [4]:
assert m.weighted_gradient([1,3], [2,6]) == 2.5
assert m.linear_prefix([1,2,1], [4,1,3])[-1] == m.linear_prefix([1,2,1], [3,1,4])[-1]
spec = m.speculative_correction([.6,.4], [.8,.2])
accepted = [q*a for q,a in zip([.8,.2], spec["accept"])]
rejection = 1-sum(accepted)
recovered = [a+rejection*r for a,r in zip(accepted,spec["residual"])]
assert all(math.isclose(a,b) for a,b in zip(recovered,[.6,.4]))
print("Changed-condition checks: weighted accumulation, order collision, residual correction passed.")
print("Calibration confusion:", actual["calibration"])


Changed-condition checks: weighted accumulation, order collision, residual correction passed.
Calibration confusion: {'substring': {'tp': 3, 'fp': 2, 'tn': 1, 'fn': 0}, 'strict': {'tp': 1, 'fp': 0, 'tn': 3, 'fn': 2}}
